In [ ]:
-- ============================================================
-- 01_create_materialized_views.ipynb
-- Define el modelo dimensional completo sobre tablas Gold
-- Ejecutar UNA SOLA VEZ o ante cambios de esquema
-- ============================================================

-- Tabla de hechos
CREATE OR REPLACE MATERIALIZED VIEW fintech_finpay.gold.fact_transactions
AS
SELECT
    t.transaction_id,
    t.user_id,
    t.merchant_id,
    t.channel,
    t.transaction_type,
    t.amount,
    t.currency,
    t.transaction_date,
    t.status,
    t.reference_id,
    k.tasa_reversa,
    k.score_riesgo
FROM fintech_finpay.silver.transactions t
LEFT JOIN fintech_finpay.gold.risk_kpis k
    ON t.merchant_id = k.merchant_id
    AND t.channel = k.channel
    AND t.transaction_date = k.fecha;

-- Dimensión comercio
CREATE OR REPLACE MATERIALIZED VIEW fintech_finpay.gold.dim_merchant
AS
SELECT
    merchant_id,
    merchant_name,
    category,
    country,
    affiliation_date,
    status,
    risk_level
FROM fintech_finpay.silver.merchants;

-- Dimensión usuario
CREATE OR REPLACE MATERIALIZED VIEW fintech_finpay.gold.dim_user
AS
SELECT
    user_id,
    full_name,
    document_id,
    email,
    phone,
    country,
    segment,
    registration_date
FROM fintech_finpay.silver.users;

-- Dimensión canal
CREATE OR REPLACE MATERIALIZED VIEW fintech_finpay.gold.dim_channel
AS
SELECT DISTINCT
    channel,
    CASE channel
        WHEN 'web' THEN 'Web Browser'
        WHEN 'app' THEN 'Mobile App'
        WHEN 'pos' THEN 'Punto de Venta Físico'
    END AS channel_description
FROM fintech_finpay.silver.transactions;

-- Dimensión fecha
CREATE OR REPLACE MATERIALIZED VIEW fintech_finpay.gold.dim_date
AS
SELECT DISTINCT
    transaction_date                                    AS date_key,
    YEAR(transaction_date)                              AS anio,
    QUARTER(transaction_date)                           AS trimestre,
    MONTH(transaction_date)                             AS mes,
    WEEKOFYEAR(transaction_date)                        AS semana,
    DAYOFMONTH(transaction_date)                        AS dia,
    DAYOFWEEK(transaction_date)                         AS dia_semana,
    DATE_FORMAT(transaction_date, 'EEEE')               AS nombre_dia,
    DATE_FORMAT(transaction_date, 'MMMM')               AS nombre_mes
FROM fintech_finpay.silver.transactions
WHERE transaction_date IS NOT NULL;